# BloodMNIST Experiments (CNN or MLP)
Use this notebook as the runner. Shared code lives in the `ml/` package.

In [ ]:
from ml.data import build_bloodmnist_dataloaders
from ml.models import CNN, MLP
from ml.setup import get_device, make_criterion, make_optimizer
from ml.training import EarlyStopping, train_with_validation, evaluate_accuracy

In [ ]:
BATCH_SIZE = 128
data_bundle = build_bloodmnist_dataloaders(batch_size=BATCH_SIZE, download=True)

train_loader = data_bundle['train_loader']
val_loader = data_bundle['val_loader']
test_loader = data_bundle['test_loader']
n_channels = data_bundle['n_channels']
n_classes = data_bundle['n_classes']

print(f'n_channels={n_channels}, n_classes={n_classes}')

n_channels=3, n_classes=8


In [ ]:
MODEL_NAME = 'mlp'   # choose: 'cnn' or 'mlp'
OPTIMIZER_NAME = 'adam'  # choose: 'adam' or 'sgd'
LEARNING_RATE = 0.0001 # 0.01 for sgd
MOMENTUM = 0.9
DROPOUT = 0.3
PATIENCE = 50
EARLY_STOPIING_DELTA = 0.0002
WEIGHT_DECAY = 0.0005

device = get_device()

if MODEL_NAME == 'cnn':
    model = CNN(in_channels=n_channels, num_classes=n_classes, dropout=DROPOUT).to(device)
elif MODEL_NAME == 'mlp':
    model = MLP(input_dim=n_channels * 28 * 28, num_classes=n_classes, dropout=DROPOUT).to(device)
else:
    raise ValueError('MODEL_NAME must be cnn or mlp')

criterion = make_criterion('cross_entropy')
optimizer = make_optimizer(
    model,
    name=OPTIMIZER_NAME,
    lr=LEARNING_RATE,
    momentum=MOMENTUM,
    weight_decay=WEIGHT_DECAY,
)
early_stopping = EarlyStopping(PATIENCE, EARLY_STOPIING_DELTA)

print(f'Device: {device}')
print(f'Model: {MODEL_NAME}')
print(f'Optimizer: {OPTIMIZER_NAME}, lr={LEARNING_RATE}')

Device: cpu
Model: mlp
Optimizer: adam, lr=0.0003
Compile model: False


In [ ]:
history = train_with_validation(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    device=device,
    epochs=500,
    early_stopping=early_stopping,
)

print('Stopped epoch:', history['stopped_epoch'])
print('Best epoch:', history['best_epoch'])

Epoch 1	Training Loss: 1.7869	Validation Loss: 1.4593
Epoch 10	Training Loss: 0.8004	Validation Loss: 0.6924
Epoch 20	Training Loss: 0.6391	Validation Loss: 0.5839
Epoch 30	Training Loss: 0.5795	Validation Loss: 0.5257
Epoch 40	Training Loss: 0.5346	Validation Loss: 0.4654
Epoch 50	Training Loss: 0.5055	Validation Loss: 0.4387
Epoch 60	Training Loss: 0.4853	Validation Loss: 0.4273
Epoch 70	Training Loss: 0.4683	Validation Loss: 0.4230
Epoch 80	Training Loss: 0.4469	Validation Loss: 0.4171
Epoch 90	Training Loss: 0.4290	Validation Loss: 0.4043
Epoch 100	Training Loss: 0.4135	Validation Loss: 0.3922
Epoch 110	Training Loss: 0.3990	Validation Loss: 0.3852
Epoch 120	Training Loss: 0.3954	Validation Loss: 0.3674
Epoch 130	Training Loss: 0.3960	Validation Loss: 0.3524
Epoch 140	Training Loss: 0.3895	Validation Loss: 0.3702
Epoch 150	Training Loss: 0.3878	Validation Loss: 0.4040
Epoch 160	Training Loss: 0.3623	Validation Loss: 0.3628
Stopped epoch: 160
Best epoch: 130


In [5]:
test_accuracy = evaluate_accuracy(model, test_loader, device)
print(f'Test accuracy: {test_accuracy:.2f}%')

Test accuracy: 85.33%
